# EDA to MLflow Pipeline

In [45]:
import pandas as pd
import sqlite3
import os

from rich.logging import RichHandler
import logging

In [46]:
# Configure logging with Rich
logging.basicConfig(
    level="INFO",
    format="%(message)s",
    datefmt="[%X]",
    handlers=[RichHandler(rich_tracebacks=True)]
)

logger = logging.getLogger("rich")

In [47]:
# 1) Load raw CSV Files
# 1) Load raw CSV Files
data_dir = os.path.join("data", "Corrected-Data")

patients_path = os.path.join(data_dir, "patients.csv")
visits_path = os.path.join(data_dir, "visits.csv")
billing_path = os.path.join(data_dir, "billing.csv")

for path in (patients_path, visits_path, billing_path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing data file: {path}")

patients = pd.read_csv(patients_path)
visits = pd.read_csv(visits_path)
billing = pd.read_csv(billing_path)

logger.info("patients shape: %s", patients.shape)
logger.info("visits shape: %s", visits.shape)
logger.info("billing shape: %s", billing.shape)




[18:19:54] INFO     patients shape: (5000, 7)                                                      ]8;id=6513739;file:///tmp/ipykernel_1864490/1631273376.py\1631273376.py]8;;\:]8;id=6513740;file:///tmp/ipykernel_1864490/1631273376.py#17\17]8;;\

           INFO     visits shape: (25000, 8)                                                       ]8;id=6513745;file:///tmp/ipykernel_1864490/1631273376.py\1631273376.py]8;;\:]8;id=6513746;file:///tmp/ipykernel_1864490/1631273376.py#18\18]8;;\

           INFO     billing shape: (25000, 7)                                                      ]8;id=6513751;file:///tmp/ipykernel_1864490/1631273376.py\1631273376.py]8;;\:]8;id=6513752;file:///tmp/ipykernel_1864490/1631273376.py#19\19]8;;\

In [48]:

# Create db folder, if it doesn't exits
os.makedirs("./db", exist_ok=True)

# Connect to sqlite and then create db
conn = sqlite3.connect("./db/hospital.db")
cursor = conn.cursor()
logger.info("Database connected: %s", conn)

           INFO     Database connected: <sqlite3.Connection object at 0x7d0b7f28b880>               ]8;id=6513757;file:///tmp/ipykernel_1864490/2370726886.py\2370726886.py]8;;\:]8;id=6513758;file:///tmp/ipykernel_1864490/2370726886.py#7\7]8;;\

In [49]:

# write all three dataframes into sqlite tables
patients.to_sql("patients", conn, if_exists="replace", index=False)
visits.to_sql("visits", conn, if_exists="replace", index=False)
billing.to_sql("billing", conn, if_exists="replace", index=False)

logger.info("All three tables loaded into hospital db")

           INFO     All three tables loaded into hospital db                                        ]8;id=6513763;file:///tmp/ipykernel_1864490/3435725956.py\3435725956.py]8;;\:]8;id=6513764;file:///tmp/ipykernel_1864490/3435725956.py#6\6]8;;\

In [50]:
# 4) Quick preview of tables
logger.info("=== PATIENTS (first 3 rows) ===")
logger.info(pd.read_sql("select * from patients limit 3", conn).to_string())

logger.info("=== VISITS (first 3 rows) ===")
logger.info(pd.read_sql("select * from visits limit 3", conn).to_string())

logger.info("=== BILLING (first 3 rows) ===")
logger.info(pd.read_sql("select * from billing limit 3", conn).to_string())

           INFO     === PATIENTS (first 3 rows) ===                                                 ]8;id=6513769;file:///tmp/ipykernel_1864490/4173315521.py\4173315521.py]8;;\:]8;id=6513770;file:///tmp/ipykernel_1864490/4173315521.py#2\2]8;;\

           INFO        patient_id  age gender       city insurance_provider  chronic_flag           ]8;id=6513775;file:///tmp/ipykernel_1864490/4173315521.py\4173315521.py]8;;\:]8;id=6513776;file:///tmp/ipykernel_1864490/4173315521.py#3\3]8;;\
                    registration_date                                                                              
                    0           1   52      M  Bangalore            CareOne             1                          
                    5/6/2025                                                                                       
                    1           2   15      F     Mumbai            CareOne             0                          
                    12/27/2025                                                                                     
                    2           3   72      F  Bangalore         SecureLife             1                          
                    1/28/2025                                                                                      

           INFO     === VISITS (first 3 rows) ===                                                   ]8;id=6513781;file:///tmp/ipykernel_1864490/4173315521.py\4173315521.py]8;;\:]8;id=6513782;file:///tmp/ipykernel_1864490/4173315521.py#5\5]8;;\

           INFO        visit_id  patient_id  visit_date  department visit_type                      ]8;id=6513787;file:///tmp/ipykernel_1864490/4173315521.py\4173315521.py]8;;\:]8;id=6513788;file:///tmp/ipykernel_1864490/4173315521.py#6\6]8;;\
                    length_of_stay_hours risk_score  doctor_id                                                     
                    0         1           1  10/16/2025  Cardiology        OPD                                     
                    23.15        Low        186                                                                    
                    1         2           1   12/9/2025         ICU        ICU                                     
                    39.98       High        149                                                                    
                    2         3           1  10/29/2025          ER         ER                                     
                    20.97     Medium        122                                                                    

           INFO     === BILLING (first 3 rows) ===                                                  ]8;id=6513793;file:///tmp/ipykernel_1864490/4173315521.py\4173315521.py]8;;\:]8;id=6513794;file:///tmp/ipykernel_1864490/4173315521.py#8\8]8;;\

           INFO        bill_id  visit_id  billed_amount  approved_amount claim_status  payment_days ]8;id=6513799;file:///tmp/ipykernel_1864490/4173315521.py\4173315521.py]8;;\:]8;id=6513800;file:///tmp/ipykernel_1864490/4173315521.py#9\9]8;;\
                    billing_date                                                                                   
                    0        1         1       23550.00         21429.62         Paid           8.0                
                    2025-11-03                                                                                     
                    1        2         2       88539.01         19725.49      Pending           NaN                
                    2025-12-16                                                                                     
                    2        3         3       25949.30          1580.95     Rejected           NaN                
                    2025-11-20                                                                                     

## Operational Analytics

In [51]:
dept_workload = pd.read_sql("""
SELECT 
    department,
    COUNT(visit_id)                             AS total_visits,
    ROUND(AVG(length_of_stay_hours), 2)         AS avg_los_hours,
    ROUND(MAX(length_of_stay_hours), 2)         AS max_los_hours,
    SUM(CASE WHEN risk_score = 'High'
        THEN 1 ELSE 0 END)                      AS high_risk_visits
    FROM visits
    GROUP BY department
    ORDER BY total_visits DESC
""", conn)

logger.info(dept_workload.to_string(index=False))

# Which doctors handle the most high-risk patients?
doctor_risk = pd.read_sql("""
    SELECT 
        doctor_id,
        COUNT(visit_id)                              AS total_visits,
        SUM(CASE WHEN risk_score = 'High' 
            THEN 1 ELSE 0 END)                       AS high_risk_cases,
        ROUND(
            100.0 * SUM(CASE WHEN risk_score = 'High' 
            THEN 1 ELSE 0 END) / COUNT(visit_id), 1) AS high_risk_pct
    FROM visits
    GROUP BY doctor_id
    ORDER BY high_risk_cases DESC
    LIMIT 10
""", conn)

logger.info("Top 10 Doctors by High Risk Cases")
logger.info(doctor_risk.to_string(index=False))

# How many visits does each patient make on average?
visit_patterns = pd.read_sql("""
    SELECT 
        visit_frequency_bucket,
        COUNT(patient_id) AS num_patients
    FROM (
        SELECT 
            patient_id,
            COUNT(visit_id) AS visit_count,
            CASE 
                WHEN COUNT(visit_id) = 1     THEN '1 visit'
                WHEN COUNT(visit_id) BETWEEN 2 AND 3 THEN '2-3 visits'
                WHEN COUNT(visit_id) BETWEEN 4 AND 5 THEN '4-5 visits'
                ELSE '6+ visits'
            END AS visit_frequency_bucket
        FROM visits
        GROUP BY patient_id
    )
    GROUP BY visit_frequency_bucket
    ORDER BY num_patients DESC
""", conn)

logger.info("Patient Visit Frequency Distribution")
logger.info(visit_patterns.to_string(index=False))

# 8) Risk Score Distribution by Department
risk_by_dept = pd.read_sql("""
    SELECT 
        department,
        SUM(CASE WHEN risk_score = 'Low'    THEN 1 ELSE 0 END) AS low,
        SUM(CASE WHEN risk_score = 'Medium' THEN 1 ELSE 0 END) AS medium,
        SUM(CASE WHEN risk_score = 'High'   THEN 1 ELSE 0 END) AS high,
        COUNT(*) AS total
    FROM visits
    GROUP BY department
    ORDER BY high DESC
""", conn)

print("Risk Score Distribution by Department")
print(risk_by_dept.to_string(index=False))

           INFO      department  total_visits  avg_los_hours  max_los_hours  high_risk_visits      ]8;id=6513805;file:///tmp/ipykernel_1864490/1419670600.py\1419670600.py]8;;\:]8;id=6513806;file:///tmp/ipykernel_1864490/1419670600.py#14\14]8;;\
                        General          5757          14.01          36.54                24                      
                    Orthopedics          4474          23.02          52.99                54                      
                     Cardiology          4071          32.59          70.74               489                      
                             ER          3896          11.95          30.78               139                      
                            ICU          3415          56.86          78.42              2975                      
                      Neurology          3387          27.69          66.16               635                      

           INFO     Top 10 Doctors by High Risk Cases                                              ]8;id=6513811;file:///tmp/ipykernel_1864490/1419670600.py\1419670600.py]8;;\:]8;id=6513812;file:///tmp/ipykernel_1864490/1419670600.py#32\32]8;;\

           INFO      doctor_id  total_visits  high_risk_cases  high_risk_pct                       ]8;id=6513817;file:///tmp/ipykernel_1864490/1419670600.py\1419670600.py]8;;\:]8;id=6513818;file:///tmp/ipykernel_1864490/1419670600.py#33\33]8;;\
                           114           260               61           23.5                                       
                           130           264               60           22.7                                       
                           187           262               57           21.8                                       
                           126           244               55           22.5                                       
                           102           271               55           20.3                                       
                           189           240               54           22.5                                       
                           158           252               53           21.0                                       
                           171           260               52           20.0                                       
                           150           252               52           20.6                                       
                           194           267               51           19.1                                       

           INFO     Patient Visit Frequency Distribution                                           ]8;id=6513823;file:///tmp/ipykernel_1864490/1419670600.py\1419670600.py]8;;\:]8;id=6513824;file:///tmp/ipykernel_1864490/1419670600.py#57\57]8;;\

           INFO     visit_frequency_bucket  num_patients                                           ]8;id=6513829;file:///tmp/ipykernel_1864490/1419670600.py\1419670600.py]8;;\:]8;id=6513830;file:///tmp/ipykernel_1864490/1419670600.py#58\58]8;;\
                                 6+ visits          1816                                                           
                                4-5 visits          1776                                                           
                                2-3 visits          1267                                                           
                                   1 visit           141                                                           

Risk Score Distribution by Department
 department  low  medium  high  total
        ICU   10     430  2975   3415
  Neurology  875    1877   635   3387
 Cardiology 1063    2519   489   4071
         ER 1771    1986   139   3896
Orthopedics 3426     994    54   4474
    General 4590    1143    24   5757


Financial Analytics

In [52]:
# Revenue and claim outcomes by insurance provider.
insurance_billing = pd.read_sql("""
    SELECT 
        p.insurance_provider,
        COUNT(b.bill_id)                         AS total_claims,
        ROUND(SUM(b.billed_amount), 0)           AS total_billed,
        ROUND(AVG(b.billed_amount), 0)           AS avg_billed,
        ROUND(SUM(b.approved_amount), 0)         AS total_approved,
        SUM(CASE WHEN b.claim_status = 'Paid'     
            THEN 1 ELSE 0 END)                   AS paid,
        SUM(CASE WHEN b.claim_status = 'Pending'  
            THEN 1 ELSE 0 END)                   AS pending,
        SUM(CASE WHEN b.claim_status = 'Rejected' 
            THEN 1 ELSE 0 END)                   AS rejected
    FROM billing b
    JOIN visits  v ON b.visit_id   = v.visit_id
    JOIN patients p ON v.patient_id = p.patient_id
    GROUP BY p.insurance_provider
    ORDER BY total_billed DESC
""", conn)

logger.info("Insurance Billing Breakdown")
logger.info(insurance_billing.to_string(index=False))

rejection_analysis = pd.read_sql("""
    SELECT 
        p.insurance_provider,
        COUNT(b.bill_id)                              AS total_claims,
        SUM(CASE WHEN b.claim_status = 'Rejected' 
            THEN 1 ELSE 0 END)                        AS rejected_claims,
        ROUND(
            100.0 * SUM(CASE WHEN b.claim_status = 'Rejected' 
            THEN 1 ELSE 0 END) / COUNT(b.bill_id), 1) AS rejection_rate_pct
    FROM billing b
    JOIN visits   v ON b.visit_id   = v.visit_id
    JOIN patients p ON v.patient_id = p.patient_id
    GROUP BY p.insurance_provider
    ORDER BY rejection_rate_pct DESC
""", conn)

logger.info("Claim Rejection Rate by Insurance Provider")
logger.info(rejection_analysis.to_string(index=False))

# How much of what we bill actually gets approved?
revenue_realization = pd.read_sql("""
    SELECT 
        p.insurance_provider,
        ROUND(SUM(b.billed_amount), 0)                AS total_billed,
        ROUND(SUM(b.approved_amount), 0)              AS total_approved,
        ROUND(
            100.0 * SUM(b.approved_amount) / 
            SUM(b.billed_amount), 1)                  AS realization_rate_pct
    FROM billing b
    JOIN visits   v ON b.visit_id   = v.visit_id
    JOIN patients p ON v.patient_id = p.patient_id
    WHERE b.approved_amount IS NOT NULL
    GROUP BY p.insurance_provider
    ORDER BY realization_rate_pct DESC
""", conn)

logger.info("Revenue Realization Rate by Insurance Provider")
logger.info(revenue_realization.to_string(index=False))



           INFO     Insurance Billing Breakdown                                                    ]8;id=6513835;file:///tmp/ipykernel_1864490/2203644646.py\2203644646.py]8;;\:]8;id=6513836;file:///tmp/ipykernel_1864490/2203644646.py#22\22]8;;\

           INFO     insurance_provider  total_claims  total_billed  avg_billed  total_approved     ]8;id=6513841;file:///tmp/ipykernel_1864490/2203644646.py\2203644646.py]8;;\:]8;id=6513842;file:///tmp/ipykernel_1864490/2203644646.py#23\23]8;;\
                    paid  pending  rejected                                                                        
                            HealthPlus          6499   251723095.0     38733.0     124033289.0                     
                    3727     1557      1215                                                                        
                               CareOne          6326   246192121.0     38918.0     107084590.0                     
                    3160     1541      1625                                                                        
                            SecureLife          6197   242381528.0     39113.0     128230940.0                     
                    3723     1498       976                                                                        
                             MediCareX          5978   229000682.0     38307.0     100849491.0                     
                    3028     1500      1450                                                                        

           INFO     Claim Rejection Rate by Insurance Provider                                     ]8;id=6513847;file:///tmp/ipykernel_1864490/2203644646.py\2203644646.py]8;;\:]8;id=6513848;file:///tmp/ipykernel_1864490/2203644646.py#41\41]8;;\

           INFO     insurance_provider  total_claims  rejected_claims  rejection_rate_pct          ]8;id=6513853;file:///tmp/ipykernel_1864490/2203644646.py\2203644646.py]8;;\:]8;id=6513854;file:///tmp/ipykernel_1864490/2203644646.py#42\42]8;;\
                               CareOne          6326             1625                25.7                          
                             MediCareX          5978             1450                24.3                          
                            HealthPlus          6499             1215                18.7                          
                            SecureLife          6197              976                15.7                          

           INFO     Revenue Realization Rate by Insurance Provider                                 ]8;id=6513859;file:///tmp/ipykernel_1864490/2203644646.py\2203644646.py]8;;\:]8;id=6513860;file:///tmp/ipykernel_1864490/2203644646.py#61\61]8;;\

           INFO     insurance_provider  total_billed  total_approved  realization_rate_pct         ]8;id=6513865;file:///tmp/ipykernel_1864490/2203644646.py\2203644646.py]8;;\:]8;id=6513866;file:///tmp/ipykernel_1864490/2203644646.py#62\62]8;;\
                            SecureLife   181215374.0     128230940.0                  70.8                         
                            HealthPlus   180661516.0     124033289.0                  68.7                         
                             MediCareX   157807601.0     100849491.0                  63.9                         
                               CareOne   169349090.0     107084590.0                  63.2                         

## Data Quality Checks

In [53]:
logger.info("=== MISSING VALUES CHECK ===\n")

for table in ['patients', 'visits', 'billing']:
    df = pd.read_sql(f"SELECT * FROM {table}", conn)
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls) > 0:
        logger.warning(f"{table}:")
        logger.warning(nulls.to_string())
        logger.warning('\n')
    else:
        logger.info(f"{table}: ✓ No missing values\n")

           INFO     === MISSING VALUES CHECK ===                                                    ]8;id=6513871;file:///tmp/ipykernel_1864490/1994823528.py\1994823528.py]8;;\:]8;id=6513872;file:///tmp/ipykernel_1864490/1994823528.py#1\1]8;;\
                                                                                                                   

           INFO     patients: ✓ No missing values                                                  ]8;id=6513877;file:///tmp/ipykernel_1864490/1994823528.py\1994823528.py]8;;\:]8;id=6513878;file:///tmp/ipykernel_1864490/1994823528.py#12\12]8;;\
                                                                                                                   

           INFO     visits: ✓ No missing values                                                    ]8;id=6513883;file:///tmp/ipykernel_1864490/1994823528.py\1994823528.py]8;;\:]8;id=6513884;file:///tmp/ipykernel_1864490/1994823528.py#12\12]8;;\
                                                                                                                   

           WARNING  billing:                                                                        ]8;id=6513889;file:///tmp/ipykernel_1864490/1994823528.py\1994823528.py]8;;\:]8;id=6513890;file:///tmp/ipykernel_1864490/1994823528.py#8\8]8;;\

           WARNING  approved_amount    6156                                                         ]8;id=6513895;file:///tmp/ipykernel_1864490/1994823528.py\1994823528.py]8;;\:]8;id=6513896;file:///tmp/ipykernel_1864490/1994823528.py#9\9]8;;\
                    payment_days       9532                                                                        

           WARNING                                                                                 ]8;id=6513901;file:///tmp/ipykernel_1864490/1994823528.py\1994823528.py]8;;\:]8;id=6513902;file:///tmp/ipykernel_1864490/1994823528.py#10\10]8;;\
                                                                                                                   

In [54]:
# 13) Duplicate Patients Check
duplicate_patients = pd.read_sql("""
    SELECT 
        patient_id,
        COUNT(*) AS occurrences
    FROM patients
    GROUP BY patient_id
    HAVING COUNT(*) > 1
""", conn)

if len(duplicate_patients) == 0:
    logger.info("✓ No duplicate patient_ids found")
else:
    logger.warning(f"⚠ Found {len(duplicate_patients)} duplicate patient_ids")
    logger.warning(duplicate_patients.to_string(index=False))

           INFO     ✓ No duplicate patient_ids found                                               ]8;id=6513907;file:///tmp/ipykernel_1864490/2066681738.py\2066681738.py]8;;\:]8;id=6513908;file:///tmp/ipykernel_1864490/2066681738.py#12\12]8;;\

In [55]:
# Visits without matching patients
orphan_visits = pd.read_sql("""
    SELECT COUNT(*) AS orphan_visits
    FROM visits v
    LEFT JOIN patients p ON v.patient_id = p.patient_id
    WHERE p.patient_id IS NULL
""", conn)

# Billing without matching visits
orphan_billing = pd.read_sql("""
    SELECT COUNT(*) AS orphan_billing
    FROM billing b
    LEFT JOIN visits v ON b.visit_id = v.visit_id
    WHERE v.visit_id IS NULL
""", conn)

logger.info("Orphan visits  (no matching patient):", 
      orphan_visits['orphan_visits'][0])
logger.info("Orphan billing (no matching visit)  :", 
      orphan_billing['orphan_billing'][0])

TypeError: not all arguments converted during string formatting

In [ ]:
# Visits without matching patients
orphan_visits = pd.read_sql("""
    SELECT COUNT(*) AS orphan_visits
    FROM visits v
    LEFT JOIN patients p ON v.patient_id = p.patient_id
    WHERE p.patient_id IS NULL
""", conn)

# Billing without matching visits
orphan_billing = pd.read_sql("""
    SELECT COUNT(*) AS orphan_billing
    FROM billing b
    LEFT JOIN visits v ON b.visit_id = v.visit_id
    
    WHERE v.visit_id IS NULL
""", conn)

logger.info(f"Orphan visits  (no matching patient): {str(orphan_visits['orphan_visits'][0])}")
logger.info(f"Orphan billing (no matching visit)  : {str(orphan_billing['orphan_billing'][0])}")

[17:32:04] INFO     Orphan visits  (no matching patient): 0                                         ]8;id=6513471;file:///tmp/ipykernel_1864490/466562326.py\466562326.py]8;;\:]8;id=6513472;file:///tmp/ipykernel_1864490/466562326.py#18\18]8;;\

           INFO     Orphan billing (no matching visit)  : 0                                         ]8;id=6513478;file:///tmp/ipykernel_1864490/466562326.py\466562326.py]8;;\:]8;id=6513479;file:///tmp/ipykernel_1864490/466562326.py#19\19]8;;\

In [ ]:
# 15) Invalid LOS Check
invalid_los = pd.read_sql("""
    SELECT 
        COUNT(CASE WHEN length_of_stay_hours <= 0  THEN 1 END) AS zero_or_negative,
        COUNT(CASE WHEN length_of_stay_hours > 720 THEN 1 END) AS over_30_days,
        MIN(length_of_stay_hours)                              AS min_los,
        MAX(length_of_stay_hours)                             AS max_los,
        ROUND(AVG(length_of_stay_hours), 2)                   AS avg_los
    FROM visits
""", conn)

logger.warning("LOS Validation:")
logger.warning(invalid_los.to_string(index=False))

[17:50:04] WARNING  LOS Validation:                                                                ]8;id=6513486;file:///tmp/ipykernel_1864490/2784248900.py\2784248900.py]8;;\:]8;id=6513487;file:///tmp/ipykernel_1864490/2784248900.py#12\12]8;;\

           WARNING   zero_or_negative  over_30_days  min_los  max_los  avg_los                     ]8;id=6513493;file:///tmp/ipykernel_1864490/2784248900.py\2784248900.py]8;;\:]8;id=6513494;file:///tmp/ipykernel_1864490/2784248900.py#13\13]8;;\
                                    0             0      0.5    78.42    26.03                                     

In [ ]:
# 16) Invalid Payment Days Check
invalid_payment = pd.read_sql("""
    SELECT 
        COUNT(CASE WHEN payment_days < 0   THEN 1 END) AS negative_days,
        COUNT(CASE WHEN payment_days > 365 THEN 1 END) AS over_1_year,
        COUNT(CASE WHEN payment_days IS NULL THEN 1 END) AS nulls,
        MIN(payment_days)                               AS min_days,
        MAX(payment_days)                               AS max_days
    FROM billing
""", conn)

logger.warning("Payment Days Validation:")
logger.warning(invalid_payment.to_string(index=False))

## Export data for model building

In [56]:
# Join all three tables into one flat table for ML modeling.
# This is the single source of truth for Phase 2 EDA and Phase 3 Modeling.
model_table = pd.read_sql("""
    SELECT
        -- Patient features
        p.patient_id,
        p.age,
        p.gender,
        p.city,
        p.insurance_provider,
        p.chronic_flag,
        p.registration_date,

        -- Visit features
        v.visit_id,
        v.visit_date,
        v.department,
        v.visit_type,
        v.length_of_stay_hours,
        v.risk_score,
        v.doctor_id,

        -- Billing features
        b.bill_id,
        b.billed_amount,
        b.approved_amount,
        b.claim_status,
        b.payment_days,
        b.billing_date

    FROM visits v
    JOIN patients p ON v.patient_id = p.patient_id
    JOIN billing  b ON v.visit_id   = b.visit_id
    ORDER BY v.visit_date
""", conn)

logger.info(f"model_table shape: {model_table.shape}")
logger.info(f"Columns: {model_table.columns.tolist()}")
model_table.head(3)

# Save the model_table to CSV for Phase 2 EDA and Phase 3 Modeling
os.makedirs("../outputs", exist_ok=True)
model_table.to_csv("./outputs/model_table.csv", index=False)
logger.info(f"model_table.csv saved --> /outputs/model_table.csv")
logger.info(f"Shape: {model_table.shape}")
logger.info(f"Size: {os.path.getsize('../outputs/model_table.csv')/1024:.1f} KB")

# Close the database connection
conn.close()

[18:19:59] INFO     model_table shape: (25000, 20)                                                 ]8;id=6513915;file:///tmp/ipykernel_1864490/3787163543.py\3787163543.py]8;;\:]8;id=6513916;file:///tmp/ipykernel_1864490/3787163543.py#37\37]8;;\

           INFO     Columns: ['patient_id', 'age', 'gender', 'city', 'insurance_provider',         ]8;id=6513922;file:///tmp/ipykernel_1864490/3787163543.py\3787163543.py]8;;\:]8;id=6513923;file:///tmp/ipykernel_1864490/3787163543.py#38\38]8;;\
                    'chronic_flag', 'registration_date', 'visit_id', 'visit_date', 'department',                   
                    'visit_type', 'length_of_stay_hours', 'risk_score', 'doctor_id', 'bill_id',                    
                    'billed_amount', 'approved_amount', 'claim_status', 'payment_days',                            
                    'billing_date']                                                                                

OSError: Cannot save file into a non-existent directory: 'outputs'